# NBA Game-Prediction Dataset Builder (Leakage-Free)

**Source:** [Kaggle - Wyatt Walsh's basketball dataset](https://www.kaggle.com/datasets/wyattowalsh/basketball)
**Input:** `nba.sqlite`

This notebook produces two files:

1. `box_data.csv` — game-level box scores in the T1/T2 layout (T1=home, T2=away).
2. `matchups.csv` — both teams' **pre-game season-to-date features** for every game, ready for modeling.

### What changed vs. the previous version

The earlier version aggregated each team's stats across the **entire season** and attached those stats to every game. That's data leakage — each game's features included the outcome of that game and all future games. This version recomputes features so that for any game on date *D*, a team's features are averaged over **only that team's games in the same season strictly before *D***.

Concretely:

- `Avg_*` and `Avg_Opp_*` are season-to-date per-game averages (cumulative through the prior game).
- `Win_Pct`, `Home_Win_Pct`, `Away_Win_Pct` count only prior games.
- `Last_14_Days_Win_Pct` uses a `[D-14, D)` rolling window.
- `Pregame_Elo` (renamed from `End_Season_Elo`) is the team's Elo *entering* the game.
- `seed` is the team's win-pct rank within the season **as of just before the game date**.

The first game of each team-season has no priors, so its features are `NaN`.


## Setup

In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Config ----
DB_PATH       = "nba.sqlite"          # update if your file is named differently
OUT_DIR       = Path(".")
ELO_BASE      = 1500.0
ELO_K         = 20.0
ELO_HOME_ADV  = 100.0                 # ~3 pts of home-court advantage
ELO_REGRESS   = 0.25                  # 25% regression toward 1500 each off-season

# Per-game box stats we track for both team and opponent
STAT_COLS = ['Score','FGM','FGA','FGM3','FGA3','FTM','FTA',
             'OR','DR','Ast','TO','Stl','Blk','PF']
OPP_COLS  = [f'Opp_{s}' for s in STAT_COLS]

## Step 1 — Load games and reshape into the T1/T2 box layout

Same as before. Each row is one game; T1 is the home team and T2 is the away team. `Target_Win = 1` if T1 won.

In [ ]:
def load_games(db_path: str) -> pd.DataFrame:
    """Pull raw home/away rows from the `game` table."""
    conn = sqlite3.connect(db_path)
    q = """
        SELECT game_id, season_id, game_date,
               team_id_home, team_name_home, wl_home,
               pts_home,  fgm_home,  fga_home,
               fg3m_home, fg3a_home, ftm_home, fta_home,
               oreb_home, dreb_home, ast_home, stl_home,
               blk_home,  tov_home,  pf_home,
               team_id_away, team_name_away,
               pts_away,  fgm_away,  fga_away,
               fg3m_away, fg3a_away, ftm_away, fta_away,
               oreb_away, dreb_away, ast_away, stl_away,
               blk_away,  tov_away,  pf_away
        FROM   game
        WHERE  wl_home IS NOT NULL
          AND  pts_home IS NOT NULL
          AND  pts_away IS NOT NULL
        ORDER  BY game_date
    """
    df = pd.read_sql(q, conn)
    conn.close()
    return df

In [ ]:
def create_box_data(games: pd.DataFrame) -> pd.DataFrame:
    """Convert raw home/away rows into the T1/T2 box layout."""
    games['Season']    = games['season_id'].astype(str).str[-4:].astype(int)
    games['game_date'] = pd.to_datetime(games['game_date'])

    box = pd.DataFrame({
        'Season'      : games['Season'],
        'DayDate'     : games['game_date'],
        'GameID'      : games['game_id'],
        'T1_TeamID'   : games['team_id_home'],
        'T1_TeamName' : games['team_name_home'],
        'T2_TeamID'   : games['team_id_away'],
        'T2_TeamName' : games['team_name_away'],
        'T1_Loc'      : 'H',

        'T1_Score': games['pts_home'],
        'T1_FGM' : games['fgm_home'],  'T1_FGA' : games['fga_home'],
        'T1_FGM3': games['fg3m_home'], 'T1_FGA3': games['fg3a_home'],
        'T1_FTM' : games['ftm_home'],  'T1_FTA' : games['fta_home'],
        'T1_OR'  : games['oreb_home'], 'T1_DR'  : games['dreb_home'],
        'T1_Ast' : games['ast_home'],  'T1_TO'  : games['tov_home'],
        'T1_Stl' : games['stl_home'],  'T1_Blk' : games['blk_home'],
        'T1_PF'  : games['pf_home'],

        'T2_Score': games['pts_away'],
        'T2_FGM' : games['fgm_away'],  'T2_FGA' : games['fga_away'],
        'T2_FGM3': games['fg3m_away'], 'T2_FGA3': games['fg3a_away'],
        'T2_FTM' : games['ftm_away'],  'T2_FTA' : games['fta_away'],
        'T2_OR'  : games['oreb_away'], 'T2_DR'  : games['dreb_away'],
        'T2_Ast' : games['ast_away'],  'T2_TO'  : games['tov_away'],
        'T2_Stl' : games['stl_away'],  'T2_Blk' : games['blk_away'],
        'T2_PF'  : games['pf_away'],

        'Target_Win': (games['wl_home'] == 'W').astype(int),
    })

    num_cols = [c for c in box.columns
                if c.startswith(('T1_', 'T2_')) and c not in
                ('T1_TeamID','T1_TeamName','T1_Loc','T2_TeamID','T2_TeamName')]
    box = box.dropna(subset=num_cols).reset_index(drop=True)
    box = box.sort_values(['DayDate', 'GameID']).reset_index(drop=True)
    return box

In [ ]:
raw = load_games(DB_PATH)
print(f"Loaded {len(raw):,} raw games")

box = create_box_data(raw)
box.to_csv(OUT_DIR / "box_data.csv", index=False)
print(f"Wrote box_data.csv ({len(box):,} rows)")
box.head()

## Step 2 — Pre-game features per (game, team)

For each game and each team in that game, compute features that use only games **strictly before** this game in the **same season**. The output is one row per `(GameID, TeamID)` — i.e., 2 rows per game. We'll merge it back onto the box file in Step 3.

### Helper: stack the box into a team-perspective game log

In [ ]:
def explode_to_team_perspective(box: pd.DataFrame) -> pd.DataFrame:
    """Each game becomes 2 rows -- one from each team's perspective."""
    def side(prefix_self, prefix_opp, loc):
        d = pd.DataFrame({
            'Season' : box['Season'],
            'DayDate': box['DayDate'],
            'GameID' : box['GameID'],
            'TeamID' : box[f'{prefix_self}_TeamID'],
            'OppID'  : box[f'{prefix_opp}_TeamID'],
            'Loc'    : loc,
            'Won'    : box['Target_Win'] if prefix_self == 'T1' else 1 - box['Target_Win'],
        })
        for s in STAT_COLS:
            d[s]          = box[f'{prefix_self}_{s}']
            d[f'Opp_{s}'] = box[f'{prefix_opp}_{s}']
        return d

    return pd.concat([side('T1', 'T2', 'H'),
                      side('T2', 'T1', 'A')], ignore_index=True)


def compute_possessions(g: pd.DataFrame) -> pd.DataFrame:
    """Per-game possessions for team and opponent (Poss = FGA - OR + TO + 0.475*FTA)."""
    g = g.copy()
    g['Poss']    = g['FGA']     - g['OR']     + g['TO']     + 0.475 * g['FTA']
    g['OppPoss'] = g['Opp_FGA'] - g['Opp_OR'] + g['Opp_TO'] + 0.475 * g['Opp_FTA']
    return g

### Helper: pre-game Elo

Walks games chronologically. For each game, **records each team's Elo entering the game**, then updates Elos with the result. The previous version only stored end-of-season Elo, which leaks future games into the feature.

In [ ]:
def compute_elo_pregame(box: pd.DataFrame,
                        base: float = ELO_BASE,
                        k: float = ELO_K,
                        hca: float = ELO_HOME_ADV,
                        regress: float = ELO_REGRESS) -> pd.DataFrame:
    """One row per game with both teams' pre-game Elo. Box must be sorted chronologically."""
    elo: dict = {}
    records = []
    prev_season = None

    for _, row in box[['Season','GameID','T1_TeamID','T2_TeamID','Target_Win']].iterrows():
        season = row['Season']
        if prev_season is not None and season != prev_season:
            for tid in elo:
                elo[tid] = (1 - regress) * elo[tid] + regress * base
        prev_season = season

        t1, t2 = row['T1_TeamID'], row['T2_TeamID']
        e1, e2 = elo.get(t1, base), elo.get(t2, base)

        # snapshot BEFORE the game updates Elo
        records.append({
            'GameID'        : row['GameID'],
            'T1_TeamID'     : t1,
            'T2_TeamID'     : t2,
            'T1_Pregame_Elo': e1,
            'T2_Pregame_Elo': e2,
        })

        exp1   = 1.0 / (1.0 + 10 ** (-((e1 + hca) - e2) / 400.0))
        delta  = k * (row['Target_Win'] - exp1)
        elo[t1] = e1 + delta
        elo[t2] = e2 - delta

    return pd.DataFrame(records)

### Main computation: cumulative season-to-date features

Within each `(TeamID, Season)`, sort by date and use `cumsum() - current_value` to get totals over **prior** games only. Per-game averages divide by `Games_Played` (also a count of prior games), so the first game of each team-season comes out as `NaN`.

In [ ]:
def compute_pregame_features(box: pd.DataFrame) -> pd.DataFrame:
    tg = explode_to_team_perspective(box)
    tg = compute_possessions(tg)
    tg = tg.sort_values(['TeamID', 'Season', 'DayDate', 'GameID']).reset_index(drop=True)

    grp_keys = ['TeamID', 'Season']
    grp = tg.groupby(grp_keys)

    # ---- Games played BEFORE this one (0 for first game of season) ----
    tg['Games_Played'] = grp.cumcount()

    # ---- Cumulative totals BEFORE this game (cumsum - current) ----
    sum_cols = STAT_COLS + OPP_COLS + ['Poss', 'OppPoss']
    for c in sum_cols:
        tg[f'cum_{c}'] = grp[c].cumsum() - tg[c]
    tg['cum_Wins'] = grp['Won'].cumsum() - tg['Won']

    # ---- Per-game averages of every box stat (own + opponent) ----
    safe_games = tg['Games_Played'].replace(0, np.nan)
    for c in STAT_COLS + OPP_COLS:
        tg[f'Avg_{c}'] = tg[f'cum_{c}'] / safe_games

    # ---- Advanced metrics on cumulative totals (your formulas) ----
    safe_poss     = tg['cum_Poss'].replace(0, np.nan)
    safe_opp_poss = tg['cum_OppPoss'].replace(0, np.nan)
    safe_fga      = tg['cum_FGA'].replace(0, np.nan)
    safe_or_dr    = (tg['cum_OR'] + tg['cum_Opp_DR']).replace(0, np.nan)

    tg['Avg_Off_Eff'] = (tg['cum_Score']     / safe_poss)     * 100   # OffEff
    tg['Avg_Def_Eff'] = (tg['cum_Opp_Score'] / safe_opp_poss) * 100   # DefEff
    tg['Net_Rating']  = tg['Avg_Off_Eff'] - tg['Avg_Def_Eff']         # NetEff
    tg['EFG_Pct']     = (tg['cum_FGM'] + 0.5 * tg['cum_FGM3']) / safe_fga
    tg['TO_Rate']     = tg['cum_TO'] / safe_poss
    tg['OR_Pct']      = tg['cum_OR'] / safe_or_dr

    # ---- Win pct (overall / home / away) BEFORE current game ----
    tg['Win_Pct']         = tg['cum_Wins'] / safe_games
    tg['Overall_Win_Pct'] = tg['Win_Pct']

    tg['_is_home']  = (tg['Loc'] == 'H').astype(int)
    tg['_is_away']  = (tg['Loc'] == 'A').astype(int)
    tg['_home_won'] = tg['Won'] * tg['_is_home']
    tg['_away_won'] = tg['Won'] * tg['_is_away']
    g2 = tg.groupby(grp_keys)
    cum_hg = g2['_is_home'].cumsum()  - tg['_is_home']
    cum_hw = g2['_home_won'].cumsum() - tg['_home_won']
    cum_ag = g2['_is_away'].cumsum()  - tg['_is_away']
    cum_aw = g2['_away_won'].cumsum() - tg['_away_won']
    tg['Home_Win_Pct'] = cum_hw / cum_hg.replace(0, np.nan)
    tg['Away_Win_Pct'] = cum_aw / cum_ag.replace(0, np.nan)
    tg = tg.drop(columns=['_is_home', '_is_away', '_home_won', '_away_won'])

    # ---- Last-14-day win pct (rolling time window, current row excluded) ----
    # Manual per-group loop -- avoids the pandas quirk where groupby.apply
    # silently drops grouping columns. closed='left' excludes the current row.
    last14 = np.full(len(tg), np.nan)
    for _, idx in tg.groupby(grp_keys, sort=False).indices.items():
        sub  = tg.iloc[idx]
        s    = pd.Series(sub['Won'].values,
                         index=pd.DatetimeIndex(sub['DayDate']))
        last14[idx] = s.rolling('14D', closed='left').mean().values
    tg['Last_14_Days_Win_Pct'] = last14

    # ---- Pre-game Elo (merged from the chronological walk) ----
    elo_df = compute_elo_pregame(box)
    elo_t1 = elo_df[['GameID','T1_TeamID','T1_Pregame_Elo']].rename(
        columns={'T1_TeamID': 'TeamID', 'T1_Pregame_Elo': 'Pregame_Elo'})
    elo_t2 = elo_df[['GameID','T2_TeamID','T2_Pregame_Elo']].rename(
        columns={'T2_TeamID': 'TeamID', 'T2_Pregame_Elo': 'Pregame_Elo'})
    elo_long = pd.concat([elo_t1, elo_t2], ignore_index=True)
    tg = tg.merge(elo_long, on=['GameID', 'TeamID'], how='left')

    # ---- Seed: rank by current Win_Pct within season as of game date ----
    # Pivot Win_Pct (date x team), forward-fill so non-playing teams keep their last
    # known pre-game Win_Pct, then rank rows. No future info is used because every
    # value being ranked already excludes the current and all later games.
    seed_parts = []
    for season, sub in tg.groupby('Season'):
        pvt = (sub.pivot_table(index='DayDate', columns='TeamID',
                               values='Win_Pct', aggfunc='mean')
                  .sort_index().ffill())
        ranks = pvt.rank(axis=1, method='min', ascending=False)
        long = ranks.reset_index().melt(id_vars='DayDate',
                                        var_name='TeamID', value_name='seed')
        long['Season'] = season
        seed_parts.append(long)
    seeds = pd.concat(seed_parts, ignore_index=True)
    tg = tg.merge(seeds, on=['Season','TeamID','DayDate'], how='left')

    # ---- Keep only the columns we actually need downstream ----
    keep = (['Season','DayDate','GameID','TeamID','OppID','Loc','Won','Games_Played']
            + [f'Avg_{c}' for c in STAT_COLS + OPP_COLS]
            + ['Avg_Off_Eff','Avg_Def_Eff','Net_Rating','EFG_Pct','TO_Rate','OR_Pct',
               'Win_Pct','Overall_Win_Pct','Home_Win_Pct','Away_Win_Pct',
               'Last_14_Days_Win_Pct','Pregame_Elo','seed'])
    return tg[keep].reset_index(drop=True)

Run Step 2:

In [ ]:
pregame = compute_pregame_features(box)
print(f"Pre-game features: {len(pregame):,} rows ({len(pregame)//2:,} games x 2 teams)")
print(f"Rows with NaN features (first game of each team-season): "
      f"{pregame['Avg_Score'].isna().sum():,}")
pregame.head()

## Step 3 — Build the matchup file

Merge the pre-game features for both T1 and T2 onto each game in `box_data`. Output column order matches your spec, with one rename: **`End_Season_Elo` is now `Pregame_Elo`** (same idea, but it's the Elo entering the game, not after the season).

In [ ]:
# Same features you specified, with End_Season_Elo -> Pregame_Elo.
FEATURE_COLS = [
    'Avg_Off_Eff', 'Avg_Def_Eff', 'Net_Rating',
    'Win_Pct', 'Pregame_Elo',
    'Overall_Win_Pct', 'Home_Win_Pct', 'Away_Win_Pct',
    'Avg_Score', 'Avg_FGM', 'Avg_FGA', 'Avg_FGM3', 'Avg_FGA3',
    'Avg_FTM',   'Avg_FTA', 'Avg_OR',  'Avg_DR',
    'Avg_Ast',   'Avg_TO',  'Avg_Stl', 'Avg_Blk', 'Avg_PF',
    'Avg_Opp_Score', 'Avg_Opp_FGM', 'Avg_Opp_FGA',
    'Avg_Opp_FGM3',  'Avg_Opp_FGA3',
    'Avg_Opp_FTM',   'Avg_Opp_FTA',
    'Avg_Opp_OR',    'Avg_Opp_DR',   'Avg_Opp_Ast',
    'Avg_Opp_TO',    'Avg_Opp_Stl',  'Avg_Opp_Blk', 'Avg_Opp_PF',
    'seed', 'Last_14_Days_Win_Pct',
]


def build_matchups(box: pd.DataFrame, pregame: pd.DataFrame) -> pd.DataFrame:
    feats = pregame[['GameID', 'TeamID'] + FEATURE_COLS]

    t1 = feats.rename(columns={'TeamID': 'T1_TeamID',
                               **{c: f'T1_{c}' for c in FEATURE_COLS}})
    t2 = feats.rename(columns={'TeamID': 'T2_TeamID',
                               **{c: f'T2_{c}' for c in FEATURE_COLS}})

    m = (box[['Season','DayDate','GameID','T1_TeamID','T2_TeamID','Target_Win']]
            .merge(t1, on=['GameID', 'T1_TeamID'])
            .merge(t2, on=['GameID', 'T2_TeamID']))

    m['ID'] = (m['Season'].astype(str) + '_'
               + m['T1_TeamID'].astype(str) + '_'
               + m['T2_TeamID'].astype(str))

    ordered = (['T1_TeamID', 'T2_TeamID', 'Season', 'DayDate', 'GameID', 'ID']
               + [f'T1_{c}' for c in FEATURE_COLS]
               + [f'T2_{c}' for c in FEATURE_COLS]
               + ['Target_Win'])
    return m[ordered]

Run Step 3:

In [ ]:
matchups = build_matchups(box, pregame)
matchups.to_csv(OUT_DIR / "matchups.csv", index=False)
print(f"Wrote matchups.csv ({len(matchups):,} rows, {len(matchups.columns)} cols)")

# Optional: drop the first-game-of-season rows that have NaN features
clean = matchups.dropna(subset=[c for c in matchups.columns if c.startswith(('T1_Avg','T2_Avg'))])
print(f"Rows after dropping NaN-feature games: {len(clean):,}")
matchups.head()

## Sanity checks (no leakage)

Two quick checks that what we built is actually leakage-free:

1. **`Games_Played` starts at 0 and counts up by 1** within each `(TeamID, Season)` — the first game of every team-season has 0 prior games.
2. **Spot-check a single team's averages**: for some game where the team has played 4+ priors, `Avg_Score` should equal the mean of that team's prior games' scores.

In [ ]:
# Check 1: Games_Played starts at 0 and increments by 1
gp_check = (pregame.sort_values(['TeamID','Season','DayDate','GameID'])
                   .groupby(['TeamID','Season'])['Games_Played']
                   .agg(['min','max','count']))
print("Games_Played min should be 0 for every team-season:",
      (gp_check['min'] == 0).all())
print("Games_Played max should equal count - 1:",
      (gp_check['max'] == gp_check['count'] - 1).all())

# Check 2: spot-check Avg_Score against a manual computation
team_log = (explode_to_team_perspective(box)
            .sort_values(['TeamID','Season','DayDate','GameID']))
sample = (pregame[pregame['Games_Played'] >= 4]
          .sort_values(['TeamID','Season','DayDate','GameID'])
          .head(1).iloc[0])

prior = team_log[(team_log['TeamID'] == sample['TeamID']) &
                 (team_log['Season'] == sample['Season']) &
                 (team_log['DayDate'] < sample['DayDate'])]
print(f"\nSpot check for TeamID={sample['TeamID']}, Season={sample['Season']}, "
      f"GameID={sample['GameID']}")
print(f"  Pre-game Avg_Score in pregame:        {sample['Avg_Score']:.3f}")
print(f"  Manual mean of prior games' Score:    {prior['Score'].mean():.3f}")

## Done

You now have:

- `box_data.csv` — game-level T1/T2 box scores
- `matchups.csv` — pre-game season-to-date features for both teams + `Target_Win`

Train on `matchups.csv`. The first game of each team-season has NaN features — drop those rows or impute them depending on your model.
